In [1]:
import random
from collections import deque

CUTOFF = "CUTOFF"
FAILURE = "FAILURE"
pop_count = 0

m,n = map(int,input("Nhap so dong va cot: ").split())
room = []

x = random.randint(0, m-1)
y = random.randint(0, n-1)
print(f"Vi tri bat dau: {x},{y}")

for i in range(m):
    row = list(map(int, input().split()))
    room.append(row)

room[x][y] = 0

def print_room(room_state,x,y):
    for i in range(m):
        for j in range(n):
            if x == i and y == j:
                print("M", end=" ")
            else:
                print(room_state[i][j], end=" ")
        print()
print("Trang thai bat dau:")
print_room(room,x,y)

def is_clean(room_state):
    for row in room_state:
        if 1 in row:
            return False
    return True

def interative_deepening_search(start_room,x,y):
    start_room = tuple(tuple(row) for row in start_room)

    for depth in range(0,100):
        result = depth_limited_search(start_room,x, y, depth)

        if result != CUTOFF:
            if result == FAILURE:
                return None
            return result

    return None

def a_star_search(start_room, start_x, start_y):
    start_room = tuple(tuple(row) for row in start_room)

    counter = 0
    g_start = 0
    h_start = count_trash(start_room)
    f_start = g_start + h_start

    frontier = []
    heapq.heappush(frontier, (f_start, counter, g_start, start_room, start_x, start_y, []))

    frontier_dict = {(start_room, start_x, start_y): g_start}

    reached = {}
    pop_count = 0

    while frontier:
        f_cost, _, g_cost, current_room, current_x, current_y, path = heapq.heappop(frontier)
        pop_count += 1

        state_signature = (current_room, current_x, current_y)

        if state_signature in frontier_dict and g_cost > frontier_dict[state_signature]:
            continue

        if state_signature in frontier_dict:
            del frontier_dict[state_signature]

        if is_clean(current_room):
            return path, pop_count

        reached[state_signature] = g_cost

        for next_room, next_x, next_y, action in get_children(current_room, current_x, current_y):
            child_signature = (next_room, next_x, next_y)

            g_new = g_cost + 1
            h_m = count_trash(next_room)
            f_new = g_new + h_m

            if child_signature in reached:
                if g_new >= reached[child_signature]:
                    continue
                else:
                    del reached[child_signature]

            if child_signature in frontier_dict:
                if g_new < frontier_dict[child_signature]:
                    frontier_dict[child_signature] = g_new
                    counter += 1
                    heapq.heappush(frontier, (f_new, counter, g_new, next_room, next_x, next_y, path + [action]))
                continue

            if child_signature not in frontier_dict and child_signature not in reached:
                frontier_dict[child_signature] = g_new
                counter += 1
                heapq.heappush(frontier, (f_new, counter, g_new, next_room, next_x, next_y, path + [action]))

    return None, pop_count2

def depth_limited_search(start_room, x, y, limit):
    global pop_count
    frontier = deque([(start_room, x, y, [], 0, frozenset([(start_room, x, y)]))])
    result = FAILURE

    while frontier:
        current_room, current_x, current_y, path, depth, ancestors = frontier.pop()
        pop_count +=1

        if is_clean(current_room):
            return path

        if depth >= limit:
            result = CUTOFF
        else:
            for next_room, next_x, next_y, action in get_children(current_room,current_x,current_y):
                child_signature = (next_room, next_x, next_y)

                if child_signature not in ancestors:
                    new_path = path + [action]
                    new_ancestors = frozenset(ancestors | {child_signature})
                    frontier.append((next_room, next_x, next_y, new_path, depth + 1, new_ancestors))
    return result

def get_children(current_room,x,y):
    children = []

    def clean(x, y):
        copy_room = [list(row) for row in current_room]
        copy_room[x][y] = 0
        return tuple(tuple(row) for row in copy_room)

    if x > 0:
        children.append((clean(x-1,y), x - 1, y, "UP"))
    if x < m - 1:
        children.append((clean(x+1,y), x + 1, y, "DOWN"))
    if y > 0:
        children.append((clean(x,y-1), x, y - 1, "LEFT"))
    if y < n - 1:
        children.append((clean(x,y+1), x, y + 1, "RIGHT"))

    return children

actions = interative_deepening_search(room,x,y)

print("------Ap dung thuat toan IDS loai 1------")

if actions is None:
    print("Khong tim thay duong di!")
else:
    print(f"Tim thay giai phap! Tong so buoc di chuyen: {len(actions)}")
    print("Cac buoc thuc hien:", " -> ".join(actions))
    print("\n--- MO PHONG TUNG BUOC CHAY ---")

    current_room_state = [list(row) for row in room]
    curr_x, curr_y = x, y

    for idx, act in enumerate(actions, 1):
        print(f"Buoc {idx}: {act}")
        if act == "UP":
            curr_x -= 1
        elif act == "DOWN":
            curr_x += 1
        elif act == "LEFT":
            curr_y -= 1
        elif act == "RIGHT":
            curr_y += 1

        current_room_state[curr_x][curr_y] = 0
        print_room(current_room_state, curr_x, curr_y)

    print(f"Tong so lan lay node ra khoi stack {pop_count}")

Vi tri bat dau: 0,1
Trang thai bat dau:
1 M 1 
1 1 1 
1 1 1 
------Ap dung thuat toan IDS loai 1------
Tim thay giai phap! Tong so buoc di chuyen: 9
Cac buoc thuc hien: RIGHT -> LEFT -> LEFT -> DOWN -> RIGHT -> RIGHT -> DOWN -> LEFT -> LEFT

--- MO PHONG TUNG BUOC CHAY ---
Buoc 1: RIGHT
1 0 M 
1 1 1 
1 1 1 
Buoc 2: LEFT
1 M 0 
1 1 1 
1 1 1 
Buoc 3: LEFT
M 0 0 
1 1 1 
1 1 1 
Buoc 4: DOWN
0 0 0 
M 1 1 
1 1 1 
Buoc 5: RIGHT
0 0 0 
0 M 1 
1 1 1 
Buoc 6: RIGHT
0 0 0 
0 0 M 
1 1 1 
Buoc 7: DOWN
0 0 0 
0 0 0 
1 1 M 
Buoc 8: LEFT
0 0 0 
0 0 0 
1 M 0 
Buoc 9: LEFT
0 0 0 
0 0 0 
M 0 0 
Tong so lan lay node ra khoi stack 4785
